# 05 — DORA Vendor Risk Propagation (PyReason)

Demonstrates PyReason adapter for DORA ICT vendor compliance:
- Boolean risk labels propagating across a vendor dependency graph
- Temporal propagation over multiple timesteps
- Uncertainty/severity kept as side-channel audit context (not PyReason payloads)

Uses the **adapter-local session + runner** path (not `Store.evaluate`). This gives
direct control over the PyReason session, rules, and timestep configuration.

**Prerequisites:** [01](01_sdk_basics.ipynb)–[02](02_rules_and_derivations.ipynb).  
**Requires:** `pyreason==3.0.0` (graceful fallback if not installed).  
**Next:** [06_problog_probabilistic.ipynb](06_problog_probabilistic.ipynb)

In [1]:
from __future__ import annotations
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from factpy_kernel.adapters.pyreason.accept import accept_pyreason_session
from factpy_kernel.adapters.pyreason.rule_ext import PyReasonRuleExt
from factpy_kernel.adapters.pyreason.runner import PyReasonRunConfig, run_pyreason
from factpy_kernel.adapters.pyreason.session import PyReasonSession
from factpy_kernel.sdk import SDKStore, Entity, Field, Identity, Relationship
from factpy_kernel.sdk.compile import compile_schema_from_classes
from factpy_kernel.sdk.dsl.expr import LogicVar, Pred
from factpy_kernel.sdk.dsl.rule import Rule

start = time.time()

## 1. Schema: Vendor Dependency Graph

In [2]:
class Vendor(Entity):
    vendor_id: str = Identity(primary_key=True)
    at_risk_signal: str = Field(cardinality="single")
    contingency_gap_signal: str = Field(cardinality="single")

class VendorDependency(Relationship):
    from_entity = Vendor
    to_entity = Vendor
    critical_path: str = Field(cardinality="single")

schema_ir = compile_schema_from_classes([Vendor, VendorDependency])
print(f"[{time.time()-start:.1f}s] Schema: {len(schema_ir['predicates'])} predicates")

[0.0s] Schema: 5 predicates


## 2. Session: Boolean Seeds + Side-Channel Assessments

Severity and exit-readiness bands are kept as side-channel context,
**not** fed into PyReason as bounded seeds.

In [3]:
session = PyReasonSession(schema_ir)

side_channel_assessments = {
    "ACME_CLOUD": {"incident_severity_band": [0.80, 0.95], "exit_readiness_band": [0.45, 0.60]},
    "PAYMENTS_GATEWAY": {"incident_severity_band": [0.55, 0.70], "exit_readiness_band": [0.30, 0.50]},
    "MOBILE_BANKING_APP": {"incident_severity_band": [0.65, 0.85], "exit_readiness_band": [0.75, 0.90]},
}

with session.batch() as tx:
    acme = tx.entity(Vendor, vendor_id="ACME_CLOUD")
    payments = tx.entity(Vendor, vendor_id="PAYMENTS_GATEWAY")
    mobile = tx.entity(Vendor, vendor_id="MOBILE_BANKING_APP")

    acme.at_risk_signal.set("true", bound=[1.0, 1.0],
        meta={"source": "major_incident_triage", "mode": "boolean_seed"})
    payments.contingency_gap_signal.set("true", bound=[1.0, 1.0],
        meta={"source": "contract_review", "mode": "boolean_seed"})

    tx.relationship(VendorDependency, from_entity=acme, to_entity=payments,
        critical_path="true", bound=[1.0, 1.0], meta={"source": "dependency_mapping"})
    tx.relationship(VendorDependency, from_entity=payments, to_entity=mobile,
        critical_path="true", bound=[1.0, 1.0], meta={"source": "dependency_mapping"})
    tx.commit()

print(f"[{time.time()-start:.1f}s] Seeds: ACME=at_risk, PAYMENTS=contingency_gap")
print(f"  Edges: ACME->PAYMENTS->MOBILE")
print(f"  Annotation templates: {len(session.annotation_templates)}")

[0.0s] Seeds: ACME=at_risk, PAYMENTS=contingency_gap
  Edges: ACME->PAYMENTS->MOBILE
  Annotation templates: 20


## 3. Rules: Boolean Risk Propagation

In [4]:
x = LogicVar("x")
y = LogicVar("y")

rules = [
    Rule(id="vendor_risk_propagation", version="1.0",
        select=[Pred("vendor:at_risk_signal", x)],
        where=[Pred("vendor:at_risk_signal", y), Pred("vendor_dependency:critical_path", y, x)],
        engine_ext=PyReasonRuleExt(timestep_delay=1)),
    Rule(id="contingency_gap_propagation", version="1.0",
        select=[Pred("vendor:contingency_gap_signal", x)],
        where=[Pred("vendor:contingency_gap_signal", y), Pred("vendor_dependency:critical_path", y, x)],
        engine_ext=PyReasonRuleExt(timestep_delay=1)),
]

print("Rules:")
print("  1. at_risk(Y) + critical_path(Y,X) -> at_risk(X) [delay=1]")
print("  2. contingency_gap(Y) + critical_path(Y,X) -> contingency_gap(X) [delay=1]")

Rules:
  1. at_risk(Y) + critical_path(Y,X) -> at_risk(X) [delay=1]
  2. contingency_gap(Y) + critical_path(Y,X) -> contingency_gap(X) [delay=1]


## 4. Run PyReason

In [5]:
try:
    result = run_pyreason(session, rule_defs=rules,
        config=PyReasonRunConfig(timesteps=4, atom_trace=True))
except Exception as exc:
    result = None
    print(f"[NOTE] PyReason not available: {exc}")
    print("Install pyreason==3.0.0 in Python 3.10 for real execution.")
    print("\nExpected behavior:")
    print("  - ACME at_risk propagates: ACME->PAYMENTS(t1)->MOBILE(t2)")
    print("  - PAYMENTS contingency_gap propagates: PAYMENTS->MOBILE(t1)")

torch is not installed, model integration is disabled


/Users/zhenzhili/miniforge3/envs/factpy/lib/python3.10/site-packages/pyreason/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


Added  0 graph-attribute node facts and  2 graph_attribute edge facts.
Filtering rules based on queries
Timestep: 0
Timestep: 1
Timestep: 2
Timestep: 3
Timestep: 4

Converged at time: 4
Fixed Point iterations: 5


## 5. Results: Active Labels per Timestep

In [6]:
if result is not None:
    interp = result.interpretation.get_dict()
    print(f"[{time.time()-start:.1f}s] Complete ({result.elapsed_seconds:.1f}s)")
    print(f"  Derived: {len(result.derived_session.node_facts)} node, {len(result.derived_session.edge_facts)} edge")

    for t in sorted(interp.keys()):
        active = []
        for comp, preds in sorted(interp[t].items()):
            labels = sorted(p for p, (lo, hi) in preds.items() if lo == 1.0 and hi == 1.0)
            if labels: active.append(f"{comp}: {', '.join(labels)}")
        if active:
            print(f"\n  Timestep {t}:")
            for line in active: print(f"    {line}")

[8.8s] Complete (6.8s)
  Derived: 3 node, 0 edge


TypeError: '<' not supported between instances of 'tuple' and 'str'

## 6. Accept + Annotations

In [ ]:
if result is not None:
    target_sdk = SDKStore([Vendor])
    accept_result = accept_pyreason_session(target_sdk.ledger, result.derived_session)
    print(f"[{time.time()-start:.1f}s] Accepted: {len(accept_result.node_asrt_ids)} assertions, "
          f"{accept_result.annotation_count} annotations")

---

**Runtime explain surface:** PyReason candidates use `explain-timeline` / `explain-timeline-summary` / `explain-timeline-narrative`
(returning `CandidateProvenanceTimeline`), **not** `explain-tree` (which returns `CandidateEvidenceTree` for native/Souffle only).

**Next:** [06_problog_probabilistic.ipynb](06_problog_probabilistic.ipynb) — ProbLog probabilistic reasoning